# ETL Samsung Galaxy - Mercado Libre
## Análisis Exploratorio de Datos

Este notebook responde las 5 preguntas de negocio del desafío usando los datos cargados en BigQuery.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from google.cloud import bigquery

os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = 'Doc/meli-514816-16a92379b1d3.json'
client = bigquery.Client(project='meli-514816')
DATASET = 'meli-514816.meli_samsung'

def query(sql):
    return client.query(sql).to_dataframe()

print('Conexión a BigQuery OK')

## Resumen general

In [ ]:
resumen = query(f"""
SELECT
    COUNT(*) AS total_publicaciones,
    COUNT(DISTINCT seller_id) AS total_vendedores,
    ROUND(AVG(price_ars), 2) AS precio_promedio_ars,
    ROUND(AVG(price_usd), 2) AS precio_promedio_usd,
    MAX(job_run) AS ultima_ejecucion
FROM `{DATASET}.items`
WHERE job_run = (SELECT MAX(job_run) FROM `{DATASET}.items`)
""")
resumen

## Pregunta 1: ¿Hay algún vendedor con múltiples publicaciones?

In [ ]:
df_multi = query(f"""
SELECT
    i.seller_id,
    s.nickname,
    COUNT(*) AS cantidad_publicaciones
FROM `{DATASET}.items` i
LEFT JOIN `{DATASET}.sellers` s ON i.seller_id = s.seller_id
    AND s.job_run = (SELECT MAX(job_run) FROM `{DATASET}.sellers`)
WHERE i.job_run = (SELECT MAX(job_run) FROM `{DATASET}.items`)
GROUP BY i.seller_id, s.nickname
HAVING COUNT(*) > 1
ORDER BY cantidad_publicaciones DESC
LIMIT 10
""")

print(f'Vendedores con múltiples publicaciones: {len(df_multi)}')
df_multi

In [ ]:
if not df_multi.empty:
    fig, ax = plt.subplots(figsize=(10, 5))
    labels = df_multi['nickname'].fillna(df_multi['seller_id']).str[:20]
    ax.barh(labels, df_multi['cantidad_publicaciones'], color='#FFE600')
    ax.set_xlabel('Cantidad de publicaciones')
    ax.set_title('Top vendedores con múltiples publicaciones')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

## Pregunta 2: ¿Promedio de ventas por seller?

In [ ]:
df_ventas = query(f"""
SELECT
    s.seller_id,
    s.nickname,
    s.total_sales,
    s.level_id,
    s.power_seller_status,
    COUNT(i.item_id) AS publicaciones,
    ROUND(s.total_sales / NULLIF(COUNT(i.item_id), 0), 2) AS ventas_promedio_por_publicacion
FROM `{DATASET}.sellers` s
JOIN `{DATASET}.items` i ON s.seller_id = i.seller_id
WHERE s.job_run = (SELECT MAX(job_run) FROM `{DATASET}.sellers`)
  AND i.job_run = (SELECT MAX(job_run) FROM `{DATASET}.items`)
GROUP BY s.seller_id, s.nickname, s.total_sales, s.level_id, s.power_seller_status
ORDER BY ventas_promedio_por_publicacion DESC
LIMIT 10
""")
df_ventas

In [ ]:
if not df_ventas.empty:
    fig, ax = plt.subplots(figsize=(10, 5))
    labels = df_ventas['nickname'].fillna(df_ventas['seller_id']).str[:20]
    ax.barh(labels, df_ventas['ventas_promedio_por_publicacion'], color='#3483FA')
    ax.set_xlabel('Ventas promedio por publicación')
    ax.set_title('Top 10 sellers por ventas promedio')
    ax.invert_yaxis()
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    plt.tight_layout()
    plt.show()

## Pregunta 3: ¿Cuál es el precio promedio en dólares?

In [ ]:
df_precios = query(f"""
SELECT
    ROUND(AVG(price_usd), 2) AS precio_promedio_usd,
    ROUND(MIN(price_usd), 2) AS precio_minimo_usd,
    ROUND(MAX(price_usd), 2) AS precio_maximo_usd,
    ROUND(STDDEV(price_usd), 2) AS desvio_estandar_usd
FROM `{DATASET}.items`
WHERE job_run = (SELECT MAX(job_run) FROM `{DATASET}.items`)
  AND price_usd IS NOT NULL
""")
df_precios

In [ ]:
df_hist = query(f"""
SELECT price_usd
FROM `{DATASET}.items`
WHERE job_run = (SELECT MAX(job_run) FROM `{DATASET}.items`)
  AND price_usd IS NOT NULL
  AND price_usd < PERCENTILE_CONT(price_usd, 0.99) OVER ()
""")

if not df_hist.empty:
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(df_hist['price_usd'], bins=30, color='#3483FA', edgecolor='white')
    ax.axvline(df_precios['precio_promedio_usd'].iloc[0], color='#FFE600', linewidth=2, label=f"Promedio: USD {df_precios['precio_promedio_usd'].iloc[0]}")
    ax.set_xlabel('Precio (USD)')
    ax.set_ylabel('Cantidad de publicaciones')
    ax.set_title('Distribución de precios en USD - Samsung Galaxy S24')
    ax.legend()
    plt.tight_layout()
    plt.show()

## Pregunta 4: ¿Porcentaje de artículos con garantía?

In [ ]:
df_garantia = query(f"""
SELECT
    COUNTIF(has_warranty = TRUE)  AS con_garantia,
    COUNTIF(has_warranty = FALSE) AS sin_garantia,
    COUNT(*) AS total,
    ROUND(COUNTIF(has_warranty = TRUE) * 100.0 / COUNT(*), 2) AS porcentaje_con_garantia
FROM `{DATASET}.items`
WHERE job_run = (SELECT MAX(job_run) FROM `{DATASET}.items`)
""")
df_garantia

In [ ]:
if not df_garantia.empty:
    valores = [df_garantia['con_garantia'].iloc[0], df_garantia['sin_garantia'].iloc[0]]
    labels = ['Con garantía', 'Sin garantía']
    colores = ['#3483FA', '#E0E0E0']

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.pie(valores, labels=labels, colors=colores, autopct='%1.1f%%', startangle=90)
    ax.set_title('Porcentaje de artículos con garantía')
    plt.tight_layout()
    plt.show()

## Pregunta 5: ¿Métodos de shipping que ofrecen?

In [ ]:
df_shipping = query(f"""
SELECT
    shipping_mode,
    logistic_type,
    COUNT(*) AS cantidad,
    COUNTIF(free_shipping = TRUE) AS con_envio_gratis,
    ROUND(COUNTIF(free_shipping = TRUE) * 100.0 / COUNT(*), 2) AS pct_envio_gratis
FROM `{DATASET}.shipping`
WHERE job_run = (SELECT MAX(job_run) FROM `{DATASET}.shipping`)
GROUP BY shipping_mode, logistic_type
ORDER BY cantidad DESC
""")
df_shipping

In [ ]:
if not df_shipping.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].bar(df_shipping['logistic_type'].fillna('N/A'), df_shipping['cantidad'], color='#3483FA')
    axes[0].set_title('Cantidad por tipo de logística')
    axes[0].set_xlabel('Tipo de logística')
    axes[0].set_ylabel('Cantidad')
    axes[0].tick_params(axis='x', rotation=30)

    axes[1].bar(df_shipping['logistic_type'].fillna('N/A'), df_shipping['pct_envio_gratis'], color='#FFE600')
    axes[1].set_title('% con envío gratis por tipo de logística')
    axes[1].set_xlabel('Tipo de logística')
    axes[1].set_ylabel('% envío gratis')
    axes[1].tick_params(axis='x', rotation=30)

    plt.tight_layout()
    plt.show()